# LLM Engine — Member 1
**Role:** AI Runtime + Model Integration  
**Responsibilities:** Install & run Qwen, test prompts, Python→LLM connection, response generation pipeline, LoRA (later)

## Step 1: Install Required Libraries

In [ ]:
# Install libraries needed to load and run the Qwen model
!pip install transformers accelerate torch

## Step 2: Load the Qwen Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load Qwen2.5-1.5B-Instruct from HuggingFace
# float16 reduces memory usage (fits 16GB RAM), device_map='auto' uses GPU if available
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model loaded on: {model.device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Step 3: Test — Basic Prompt

In [ ]:
# Quick sanity check: send a simple greeting and verify the model responds
messages = [
    {"role": "user", "content": "Hello, who are you?"}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Step 4: Build the Response Generation Function

In [ ]:
def generate_response(query: str, context: str = "", max_tokens: int = 200) -> str:
    """
    Core LLM function — takes a user query and optional context, returns a generated response.

    Args:
        query:      The user's question or instruction.
        context:    Optional retrieved text (for RAG). Injected into the prompt if provided.
        max_tokens: Maximum number of new tokens to generate.

    Returns:
        Generated response string.
    """
    # Build the prompt — include context if provided (RAG-ready)
    if context:
        prompt = (
            "Answer the question using only the provided context.\n\n"
            f"Context:\n{context}\n\n"
            f"Question:\n{query}"
        )
    else:
        prompt = query

    # Format as chat message
    messages = [{"role": "user", "content": prompt}]

    # Apply the model's chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize and move to model device
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True
    )

    # Decode and return only the new tokens (strip the input prompt)
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's reply (after the user turn)
    if "assistant" in full_output.lower():
        response = full_output.split("assistant")[-1].strip()
    else:
        response = full_output.strip()

    return response

## Step 5: Test — generate_response() Without Context

In [ ]:
# Test the function with a plain query (no context / no RAG)
response = generate_response("Explain recursion simply")
print(response)

## Step 6: Test — generate_response() With Context (RAG-ready)

In [ ]:
# Simulate what happens when another team member passes retrieved context to the LLM
# In the full project, context will come from the retrieval pipeline (FAISS + embeddings)
sample_context = (
    "Recursion is a programming technique where a function calls itself "
    "to solve a smaller version of the same problem. "
    "Every recursive function must have a base case to stop the recursion."
)

query = "What is recursion and why does it need a base case?"
response = generate_response(query, context=sample_context)
print(response)

## Step 7: Final Pipeline — Integration-Ready Function
This is the function that other team members call. It accepts a query + context and returns a clean string response.

In [ ]:
def run_llm_pipeline(query: str, context: str = "") -> dict:
    """
    Final response generation pipeline — integration point for other team members.

    Input:  query (str) + optional context (str) from retrieval pipeline
    Output: dict with 'query', 'context_used', and 'response' keys
    """
    response = generate_response(query, context=context, max_tokens=300)

    return {
        "query": query,
        "context_used": bool(context),
        "response": response
    }


# --- Test the final pipeline ---
result = run_llm_pipeline(
    query="What is recursion?",
    context="Recursion is when a function calls itself with a simpler input until it reaches a base case."
)

print(f"Query        : {result['query']}")
print(f"Context used : {result['context_used']}")
print(f"Response     :\n{result['response']}")

## Step 8: LoRA Fine-tuning Setup (Placeholder — To Do Later)
> This section is reserved for LoRA adapter integration once the base pipeline is validated.

In [ ]:
# TODO (Later): Load a LoRA adapter on top of the base Qwen model
#
# !pip install peft
#
# from peft import PeftModel
#
# lora_model = PeftModel.from_pretrained(
#     model,                         # base model already loaded above
#     "path/to/your-lora-adapter"    # replace with actual adapter path
# )
#
# Then pass lora_model wherever 'model' is used in generate_response()

print("LoRA placeholder ready — uncomment and fill in adapter path when needed.")